In [1]:
from google.colab import files

uploaded = files.upload()

Saving hostel_bois.txt to hostel_bois.txt


In [2]:
import os
print(os.path.exists("/content/hostel_bois.txt"))

True


In [ ]:
import os

print(os.listdir("/content"))

['.config', 'hostel_bois.txt', 'sample_data']


In [ ]:
file_path = "/content/hostel_bois.txt"

with open(file_path, "r", encoding="utf-8") as file:
    print(file.read()[:1000])

01/04/24, 01:12 - Messages and calls are end-to-end encrypted. No one outside of this chat, not even WhatsApp, can read or listen to them. Tap to learn more.
01/04/24, 01:13 - Priya created group "Hostel Bois 4ever"
01/04/24, 01:14 - Priya added you
01/04/24, 01:17 - Rahul: scene fix
01/04/24, 01:17 - Rahul: haan
01/04/24, 01:18 - Rahul: kya scene
01/04/24, 02:13 - Rahul: abhi free hai?
01/04/24, 02:13 - Rahul: abey
01/04/24, 02:14 - Rahul: bhai sun na
01/04/24, 07:46 - Priya: I have extra notes if anyone wants
01/04/24, 09:20 - Priya: Anyone needs help with anything?
01/04/24, 10:42 - Priya: Don't forget to call home guys
01/04/24, 11:37 - Rahul: ja yaar
01/04/24, 11:38 - Rahul: kya scene hai bhai
01/04/24, 11:38 - Rahul: bhai chai pite hai
01/04/24, 11:44 - Priya: Did you eat anything today?
01/04/24, 11:52 - Priya: Everyone doing okay with studies?
01/04/24, 12:39 - Karan: Today's class was honestly the most interesting one we have had this semester, the professor went off topic com

In [ ]:
with open("/content/hostel_bois.txt", "r", encoding="utf-8") as file:
    for i, line in enumerate(file):
        if i == 10:
            break
        print(i + 1, repr(line))

1 '01/04/24, 01:12 - Messages and calls are end-to-end encrypted. No one outside of this chat, not even WhatsApp, can read or listen to them. Tap to learn more.\n'
2 '01/04/24, 01:13 - Priya created group "Hostel Bois 4ever"\n'
3 '01/04/24, 01:14 - Priya added you\n'
4 '01/04/24, 01:17 - Rahul: scene fix\n'
5 '01/04/24, 01:17 - Rahul: haan\n'
6 '01/04/24, 01:18 - Rahul: kya scene\n'
7 '01/04/24, 02:13 - Rahul: abhi free hai?\n'
8 '01/04/24, 02:13 - Rahul: abey\n'
9 '01/04/24, 02:14 - Rahul: bhai sun na\n'
10 '01/04/24, 07:46 - Priya: I have extra notes if anyone wants\n'


In [11]:

from datetime import datetime

def parse_chat(file_path):
    messages = []
    stats = {"system": 0, "media": 0, "deleted": 0}

    with open(file_path, "r", encoding="utf-8") as file:
        for raw_line in file:
            line = raw_line.rstrip("\n\r")

            if not line.strip():
                continue

            # Find the timestamp separator, allowing 1- or 2-digit days
            if " - " not in line:
                if messages:
                    messages[-1]["text"] += " " + line.strip()
                continue

            timestamp_text, rest = line.split(" - ", 1)

            try:
                timestamp = datetime.strptime(
                    timestamp_text.strip(), "%d/%m/%y, %H:%M"
                )
            except ValueError:
                if messages:
                    messages[-1]["text"] += " " + line.strip()
                else:
                    stats["system"] += 1
                continue

            # System notifications generally have no sender/message separator.
            if ": " not in rest:
                stats["system"] += 1
                continue

            sender, message_text = rest.split(": ", 1)

            # Count special message types separately.
            if message_text == "<Media omitted>":
                stats["media"] += 1
            if message_text == "This message was deleted":
                stats["deleted"] += 1

            messages.append({
                "timestamp": timestamp_text.strip(),
                "sender": sender,
                "text": message_text
            })

    return messages, stats


messages, parser_stats = parse_chat("/content/hostel_bois.txt")
participants = sorted(set(m["sender"] for m in messages))

print("Parsed messages:", len(messages))
print("Participants:", participants)
print("System lines:", parser_stats["system"])
print("Media entries:", parser_stats["media"])
print("Deleted entries:", parser_stats["deleted"])

print("\nFirst 5 parsed messages:")
for message in messages[:5]:
    print(message)


Parsed messages: 3174
Participants: ['Aman', 'Karan', 'Neha', 'Priya', 'Rahul', 'Vikas']
System lines: 4
Media entries: 32
Deleted entries: 15

First 5 parsed messages:
{'timestamp': '01/04/24, 01:17', 'sender': 'Rahul', 'text': 'scene fix'}
{'timestamp': '01/04/24, 01:17', 'sender': 'Rahul', 'text': 'haan'}
{'timestamp': '01/04/24, 01:18', 'sender': 'Rahul', 'text': 'kya scene'}
{'timestamp': '01/04/24, 02:13', 'sender': 'Rahul', 'text': 'abhi free hai?'}
{'timestamp': '01/04/24, 02:13', 'sender': 'Rahul', 'text': 'abey'}


In [12]:

from datetime import datetime

def get_group_overview(messages):
    if not messages:
        print("No messages found.")
        return

    counts = {}
    timestamps = []

    for message in messages:
        sender = message["sender"]
        counts[sender] = counts.get(sender, 0) + 1

        timestamp = datetime.strptime(
            message["timestamp"], "%d/%m/%y, %H:%M"
        )
        timestamps.append(timestamp)

    first_date = min(timestamps)
    last_date = max(timestamps)
    total = len(messages)

    print("=" * 55)
    print("              GROUP OVERVIEW")
    print("=" * 55)
    print(f"Total messages : {total:,}")
    print(f"Participants   : {len(counts)}")
    print(f"First message  : {first_date:%d %B %Y, %H:%M}")
    print(f"Last message   : {last_date:%d %B %Y, %H:%M}")
    print(
        f"Date range     : "
        f"{(last_date.date() - first_date.date()).days + 1} days"
    )

    print("\nMESSAGES PER PERSON")
    print("-" * 55)

    for person, count in sorted(
        counts.items(), key=lambda item: item[1], reverse=True
    ):
        percentage = count / total * 100
        bar = "#" * int(percentage / 2)

        print(
            f"{person:<10} {bar:<25} "
            f"{count:>5} ({percentage:5.1f}%)"
        )

    return counts, first_date, last_date


overview = get_group_overview(messages)


              GROUP OVERVIEW
Total messages : 3,174
Participants   : 6
First message  : 01 April 2024, 01:17
Last message   : 30 May 2024, 23:31
Date range     : 60 days

MESSAGES PER PERSON
-------------------------------------------------------
Rahul      ###############             953 ( 30.0%)
Priya      ###########                 718 ( 22.6%)
Neha       ##########                  635 ( 20.0%)
Aman       #######                     490 ( 15.4%)
Karan      #####                       354 ( 11.2%)
Vikas                                   24 (  0.8%)


In [13]:

from datetime import datetime

day_counts = {}
hour_counts = {}

for message in messages:
    timestamp = datetime.strptime(
        message["timestamp"], "%d/%m/%y, %H:%M"
    )

    day = timestamp.strftime("%d/%m/%y")
    hour = timestamp.hour

    day_counts[day] = day_counts.get(day, 0) + 1
    hour_counts[hour] = hour_counts.get(hour, 0) + 1

busiest_day = max(day_counts, key=day_counts.get)
busiest_hour = max(hour_counts, key=hour_counts.get)

print("BUSIEST DAY")
print(f"{busiest_day}: {day_counts[busiest_day]} messages")

print("\nBUSIEST HOUR")
print(
    f"{busiest_hour:02d}:00–{busiest_hour:02d}:59 — "
    f"{hour_counts[busiest_hour]} messages"
)

print("\nMESSAGES BY HOUR")
for hour in range(24):
    count = hour_counts.get(hour, 0)
    bar = "#" * (count // 5)
    print(f"{hour:02d}:00 | {bar} ({count})")


BUSIEST DAY
04/05/24: 76 messages

BUSIEST HOUR
18:00–18:59 — 248 messages

MESSAGES BY HOUR
00:00 | ########### (57)
01:00 | ################ (82)
02:00 | ################ (83)
03:00 | ############### (77)
04:00 | ###################### (110)
05:00 | ##### (29)
06:00 | ###### (33)
07:00 | ########### (55)
08:00 | ######################## (122)
09:00 | ############################## (151)
10:00 | ################################ (160)
11:00 | ###################### (114)
12:00 | ###################################### (193)
13:00 | ############################### (159)
14:00 | ################################ (162)
15:00 | ########################## (131)
16:00 | ##################################### (189)
17:00 | ################################## (173)
18:00 | ################################################# (248)
19:00 | ############################################# (228)
20:00 | ################################# (166)
21:00 | ################################### (177)
22:00 | ######

In [14]:

import numpy as np
from datetime import datetime

participants = ["Rahul", "Priya", "Aman", "Karan", "Neha", "Vikas"]

# 6 participants × 24 hours
heatmap = np.zeros((6, 24), dtype=int)

for message in messages:
    sender = message["sender"]
    timestamp = datetime.strptime(
        message["timestamp"], "%d/%m/%y, %H:%M"
    )

    if sender in participants:
        row = participants.index(sender)
        hour = timestamp.hour
        heatmap[row, hour] += 1

print("ACTIVITY HEATMAP")
print("Person     " + " ".join(f"{h:02d}" for h in range(24)))

for i, person in enumerate(participants):
    print(f"{person:<10}", end=" ")
    for count in heatmap[i]:
        print("#" * min(count // 3, 9) or ".", end=" ")
    print()

print("\nMatrix shape:", heatmap.shape)
print("Total counted messages:", int(np.sum(heatmap)))


ACTIVITY HEATMAP
Person     00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23
Rahul      # ##### ##### ##### ####### ### ##### ##### ######## ##### ######## ##### ######### ######### ######### ######### ######### ######### ######### ######### ######### ######### ######### ######### 
Priya      . . . . . . #### ###### ######### ######### ######### ######### ######### ######### ######### ######### ######### ######### ######### ######### ######### ######### ###### ### 
Aman       ######### ######### ######### ######### ######### . . . . . . . . . #### ### ###### ## ##### ## #### ### . ######### 
Karan      . . . . . . . # #### ##### ###### ##### ######### ######## ######### ######### ######### ######### ######## ######### ####### #### ### ## 
Neha       . . . . . ###### # #### ######### ######### ######### ####### ######### ######### ######### ### ######### ######### ######### ######### ######### ######### ######### ######### 
Vikas      . . . . . . . . # . . . . . .

In [ ]:

# FEATURE 5: TOP WORDS ANALYSIS

stop_words = {
    "the", "is", "a", "an", "and", "or", "to", "of", "in",
    "on", "for", "it", "this", "that", "are", "was", "were",
    "be", "been", "am", "i", "you", "he", "she", "we", "they",
    "me", "my", "your", "our", "their", "so", "but", "if",
    "at", "as", "with", "from", "have", "has", "had", "do",
    "does", "did", "not", "no", "yes", "just", "ya", "hai",
    "hain", "ho", "ka", "ki", "ke", "ko", "kya", "hai", "na",
    "bhai", "abey", "yaar"
}

word_counts = {}

for message in messages:
    text = message["text"].lower()


    # Clean media placeholders and common contractions
    text = text.replace("<media omitted>", " ")
    text = text.replace("don't", "do not")
    text = text.replace("doesn't", "does not")
    text = text.replace("didn't", "did not")
    text = text.replace("it's", "it is")
    text = text.replace("that's", "that is")
    text = text.replace("what's", "what is")
    text = text.replace("i'm", "i am")
    text = text.replace("you're", "you are")
    text = text.replace("we're", "we are")
    text = text.replace("they're", "they are")
    text = text.replace("can't", "cannot")
    text = text.replace("won't", "will not")

    for punctuation in ".,!?;:()[]{}\"'":
        text = text.replace(punctuation, " ")

    words = text.split()

    for word in words:
        if word not in stop_words and len(word) > 1:
           word_counts[word] = word_counts.get(word, 0) + 1


# Sort by frequency, highest first
sorted_words = sorted(
    word_counts.items(),
    key=lambda item: item[1],
    reverse=True
)

print("=" * 45)
print("          TOP 20 WORDS")
print("=" * 45)

for rank, (word, count) in enumerate(sorted_words[:20], start=1):
    print(f"{rank:2}. {word:<20} {count:>5}")


          TOP 20 WORDS
 1. how                    321
 2. guys                   318
 3. today                  292
 4. about                  274
 5. his                    217
 6. everyone               203
 7. which                  202
 8. telling                179
 9. up                     172
10. anyone                 157
11. one                    157
12. started                150
13. scene                  145
14. entire                 145
15. please                 141
16. what                   123
17. now                    121
18. everything             121
19. came                   116
20. sleep                  112


In [15]:

# FEATURE 6: RESPONSE TIMES AND SILENT STREAKS

from datetime import datetime, timedelta

# Sort messages chronologically
timed_messages = []

for message in messages:
    timestamp = datetime.strptime(
        message["timestamp"], "%d/%m/%y, %H:%M"
    )

    timed_messages.append({
        "timestamp": timestamp,
        "sender": message["sender"],
        "text": message["text"]
    })

timed_messages.sort(key=lambda item: item["timestamp"])

# Approximate response times: next message from a different sender
response_minutes = {}

for i in range(1, len(timed_messages)):
    previous = timed_messages[i - 1]
    current = timed_messages[i]

    if current["sender"] != previous["sender"]:
        minutes = (
            current["timestamp"] - previous["timestamp"]
        ).total_seconds() / 60

        sender = current["sender"]

        if sender not in response_minutes:
            response_minutes[sender] = []

        response_minutes[sender].append(minutes)

print("=" * 55)
print("       AVERAGE RESPONSE TIME BY PERSON")
print("=" * 55)

for person in sorted(response_minutes):
    times = response_minutes[person]
    average = sum(times) / len(times)
    print(f"{person:<10} {average:8.1f} minutes ({len(times)} transitions)")

# Count active days and longest silent streak
first_day = timed_messages[0]["timestamp"].date()
last_day = timed_messages[-1]["timestamp"].date()
total_days = (last_day - first_day).days + 1

active_days = {person: set() for person in participants}

for message in timed_messages:
    active_days[message["sender"]].add(
        message["timestamp"].date()
    )

print("\n" + "=" * 55)
print("       SILENT DAYS AND LONGEST SILENT STREAK")
print("=" * 55)

for person in participants:
    days = active_days[person]
    silent_days = total_days - len(days)

    longest_streak = 0
    current_streak = 0

    for offset in range(total_days):
        day = first_day + timedelta(days=offset)

        if day not in days:
            current_streak += 1
            longest_streak = max(longest_streak, current_streak)
        else:
            current_streak = 0

    print(
        f"{person:<10} Silent days: {silent_days:2}/{total_days} | "
        f"Longest streak: {longest_streak} days"
    )


       AVERAGE RESPONSE TIME BY PERSON
Aman           55.4 minutes (180 transitions)
Karan          36.6 minutes (287 transitions)
Neha           39.4 minutes (247 transitions)
Priya          42.0 minutes (433 transitions)
Rahul          34.9 minutes (210 transitions)
Vikas          46.3 minutes (23 transitions)

       SILENT DAYS AND LONGEST SILENT STREAK
Rahul      Silent days:  0/60 | Longest streak: 0 days
Priya      Silent days:  0/60 | Longest streak: 0 days
Aman       Silent days:  0/60 | Longest streak: 0 days
Karan      Silent days:  0/60 | Longest streak: 0 days
Neha       Silent days:  0/60 | Longest streak: 0 days
Vikas      Silent days: 44/60 | Longest streak: 11 days


In [9]:

# FEATURE 7: PERSONALITY ARCHETYPES

from datetime import datetime

# Prepare per-person statistics
person_stats = {}

for person in participants:
    person_stats[person] = {
        "messages": 0,
        "total_words": 0,
        "burst_messages": 0,
        "night_messages": 0,
        "caring_messages": 0,
        "dramatic_messages": 0,
        "question_messages": 0,
        "active_days": set()
    }

caring_words = [
    "help", "care", "eat", "home", "okay", "notes",
    "study", "studies", "sleep", "health", "need"
]

for message in messages:
    person = message["sender"]
    text = message["text"]
    lower_text = text.lower()
    words = lower_text.split()
    timestamp = datetime.strptime(
        message["timestamp"], "%d/%m/%y, %H:%M"
    )

    stats = person_stats[person]
    stats["messages"] += 1
    stats["total_words"] += len(words)
    stats["active_days"].add(timestamp.date())

    if timestamp.hour >= 23 or timestamp.hour <= 4:
        stats["night_messages"] += 1

    if "?" in text:
        stats["question_messages"] += 1

    if any(word in lower_text for word in caring_words):
        stats["caring_messages"] += 1

    # All-caps or multiple exclamation marks
    letters = [char for char in text if char.isalpha()]
    all_caps = (
        len(letters) >= 3
        and all(char.isupper() for char in letters)
    )
    if all_caps or text.count("!") >= 2:
        stats["dramatic_messages"] += 1

# Count burst messages: consecutive messages by same sender
for i in range(1, len(timed_messages)):
    previous = timed_messages[i - 1]
    current = timed_messages[i]

    if current["sender"] == previous["sender"]:
        person_stats[current["sender"]]["burst_messages"] += 1

total_days = 60

print("=" * 75)
print("                 GROUPDNA PERSONALITY REPORT")
print("=" * 75)

for person in participants:
    stats = person_stats[person]
    total = stats["messages"]

    avg_words = stats["total_words"] / total if total else 0
    night_pct = stats["night_messages"] / total * 100 if total else 0
    caring_pct = stats["caring_messages"] / total * 100 if total else 0
    dramatic_pct = stats["dramatic_messages"] / total * 100 if total else 0
    burst_pct = stats["burst_messages"] / total * 100 if total else 0
    silent_pct = (
        (total_days - len(stats["active_days"])) / total_days * 100
    )

       # Choose an archetype using clear, rule-based thresholds
    if silent_pct >= 60:
        archetype = "GHOST"
    elif night_pct >= 60:
        archetype = "NIGHT OWL"
    elif avg_words >= 20:
        archetype = "STORYTELLER"
    elif dramatic_pct >= 30:
        archetype = "DRAMA QUEEN"
    elif caring_pct >= 25:
        archetype = "GROUP MOM"
    elif burst_pct >= 50:
        archetype = "SPAMMER"
    elif stats["question_messages"] / total * 100 >= 20:
        archetype = "QUESTION MASTER"
    else:
        archetype = "COMEDIAN"

    print(f"\n{person}: {archetype}")
    print(f"  Messages: {total}")
    print(f"  Average words/message: {avg_words:.1f}")
    print(f"  Night messages: {night_pct:.1f}%")
    print(f"  Caring-keyword messages: {caring_pct:.1f}%")
    print(f"  Dramatic messages: {dramatic_pct:.1f}%")
    print(f"  Silent days: {silent_pct:.1f}%")
    print(f"  Burst-message rate: {burst_pct:.1f}%")


                 GROUPDNA PERSONALITY REPORT

Rahul: SPAMMER
  Messages: 953
  Average words/message: 2.6
  Night messages: 13.4%
  Caring-keyword messages: 0.0%
  Dramatic messages: 0.0%
  Silent days: 0.0%
  Burst-message rate: 77.9%

Priya: GROUP MOM
  Messages: 718
  Average words/message: 5.0
  Night messages: 1.3%
  Caring-keyword messages: 45.1%
  Dramatic messages: 0.0%
  Silent days: 0.0%
  Burst-message rate: 39.7%

Aman: NIGHT OWL
  Messages: 490
  Average words/message: 5.0
  Night messages: 79.8%
  Caring-keyword messages: 17.8%
  Dramatic messages: 0.0%
  Silent days: 0.0%
  Burst-message rate: 63.3%

Karan: STORYTELLER
  Messages: 354
  Average words/message: 55.7
  Night messages: 2.3%
  Caring-keyword messages: 18.1%
  Dramatic messages: 0.0%
  Silent days: 0.0%
  Burst-message rate: 18.9%

Neha: DRAMA QUEEN
  Messages: 635
  Average words/message: 5.3
  Night messages: 4.7%
  Caring-keyword messages: 8.0%
  Dramatic messages: 62.2%
  Silent days: 0.0%
  Burst-message 

In [16]:

# FEATURE 8: FINAL GROUPDNA REPORT

from datetime import datetime

print("=" * 65)
print("                 GROUPDNA — FINAL REPORT")
print("=" * 65)

print("\n1. GROUP SUMMARY")
print("-" * 65)
print("Group: Hostel Bois 4ever")
print(f"Total messages: {len(messages):,}")
print(f"Participants: {len(participants)}")
print("Analysis period: 01 April 2024 – 30 May 2024 (60 days)")
print(f"System messages: {parser_stats['system']}")
print(f"Media entries: {parser_stats['media']}")
print(f"Deleted-message entries: {parser_stats['deleted']}")

print("\n2. MOST ACTIVE PARTICIPANTS")
print("-" * 65)

for person, count in sorted(
    overview[0].items(),
    key=lambda item: item[1],
    reverse=True
):
    print(f"{person:<10} {count:>5} messages ({count / len(messages) * 100:.1f}%)")

print("\n3. PEAK ACTIVITY")
print("-" * 65)
print(f"Busiest day: {busiest_day} ({day_counts[busiest_day]} messages)")
print(
    f"Busiest hour: {busiest_hour:02d}:00–"
    f"{busiest_hour:02d}:59 ({hour_counts[busiest_hour]} messages)"
)

print("\n4. SILENCE ANALYSIS")
print("-" * 65)

for person in participants:
    silent_days = total_days - len(active_days[person])
    silent_pct = silent_days / total_days * 100
    print(f"{person:<10} {silent_days:>2} silent days ({silent_pct:.1f}%)")

print("\n5. PERSONALITY ARCHETYPES")
print("-" * 65)

for person in participants:
    stats = person_stats[person]
    total = stats["messages"]

    avg_words = stats["total_words"] / total if total else 0
    night_pct = stats["night_messages"] / total * 100 if total else 0
    caring_pct = stats["caring_messages"] / total * 100 if total else 0
    dramatic_pct = stats["dramatic_messages"] / total * 100 if total else 0
    silent_pct = (
        (total_days - len(stats["active_days"])) / total_days * 100
    )
    burst_pct = stats["burst_messages"] / total * 100 if total else 0

    if silent_pct >= 60:
        archetype = "GHOST"
    elif night_pct >= 60:
        archetype = "NIGHT OWL"
    elif avg_words >= 20:
        archetype = "STORYTELLER"
    elif dramatic_pct >= 30:
        archetype = "DRAMA QUEEN"
    elif caring_pct >= 25:
        archetype = "GROUP MOM"
    elif burst_pct >= 50:
        archetype = "SPAMMER"
    elif stats["question_messages"] / total * 100 >= 20:
        archetype = "QUESTION MASTER"
    else:
        archetype = "COMEDIAN"

    print(f"{person:<10} → {archetype}")

print("\n6. KEY OBSERVATIONS")
print("-" * 65)
print("- Rahul contributes the largest share of messages.")
print("- Group activity peaks during the early evening.")
print("- Vikas has the highest number of silent days.")
print("- The archetypes are based on message patterns, not personal diagnoses.")

print("\n" + "=" * 65)
print("END OF GROUPDNA REPORT")
print("=" * 65)


                 GROUPDNA — FINAL REPORT

1. GROUP SUMMARY
-----------------------------------------------------------------
Group: Hostel Bois 4ever
Total messages: 3,174
Participants: 6
Analysis period: 01 April 2024 – 30 May 2024 (60 days)
System messages: 4
Media entries: 32
Deleted-message entries: 15

2. MOST ACTIVE PARTICIPANTS
-----------------------------------------------------------------
Rahul        953 messages (30.0%)
Priya        718 messages (22.6%)
Neha         635 messages (20.0%)
Aman         490 messages (15.4%)
Karan        354 messages (11.2%)
Vikas         24 messages (0.8%)

3. PEAK ACTIVITY
-----------------------------------------------------------------
Busiest day: 04/05/24 (76 messages)
Busiest hour: 18:00–18:59 (248 messages)

4. SILENCE ANALYSIS
-----------------------------------------------------------------
Rahul       0 silent days (0.0%)
Priya       0 silent days (0.0%)
Aman        0 silent days (0.0%)
Karan       0 silent days (0.0%)
Neha        0 

## Final Reflection

Through this GroupDNA project, I learned how to analyze WhatsApp group conversations using Python. I used lists, dictionaries, sets, loops, functions, file handling, and the datetime module to process chat data.

I calculated message counts for each participant, identified the busiest day and hour, and used NumPy to create a 6 × 24 hourly activity heatmap. I also analyzed frequently used words, estimated response times, measured silent days, and assigned rule-based personality archetypes.

One challenge was handling media entries, deleted messages, system notifications, and multiline messages. This helped me understand the importance of cleaning and validating data before analysis.

I also learned that analytical results depend on the rules used. Response-time estimates may not represent direct replies, and personality archetypes are simplified labels based on messaging patterns rather than actual personality assessments.

Overall, this project improved my Python programming, data analysis, and problem-solving skills. In the future, I would like to improve reply detection and make the analysis more accurate.

**AI assistance disclosure:** AI assistance was used for guidance with code structure, debugging, and documentation.